# Vérification de fake news par la knowledge-base method

In [3]:
import requests

url = "https://zenodo.org/record/3609356/files/groundtruth.csv?download=1"
response = requests.get(url, allow_redirects=True)

with open("groundtruth.csv", "wb") as f:
    f.write(response.content)
print("✅ Fichier groundtruth.csv téléchargé avec succès !")

✅ Fichier groundtruth.csv téléchargé avec succès !


In [4]:
import pandas as pd

dataset = pd.read_csv("groundtruth.csv")
print(dataset.head())
print(dataset.columns.to_list)
print(dataset['Verdict'].value_counts())
print(dataset.shape)

# On crée une colonne 'labels' binaire : 1 pour le niveau 1, 0 pour le reste
dataset['labels'] = dataset['Verdict'].apply(lambda x: 1 if x == 1 else 0)

# On ne garde que le texte et le nouveau label
dataset = dataset[['Text', 'labels']]

   Sentence_id                                               Text  \
0           26      You know, I saw a movie - "Crocodile Dundee."   
1           80  We're consuming 50 percent of the world's coca...   
2          129   That answer was about as clear as Boston harbor.   
3          131                          Let me help the governor.   
4          172  We've run up more debt in the last eight years...   

           Speaker   Speaker_title Speaker_party         File_id  Length  \
0      George Bush  Vice President    REPUBLICAN  1988-09-25.txt       9   
1  Michael Dukakis        Governor      DEMOCRAT  1988-09-25.txt       8   
2      George Bush  Vice President    REPUBLICAN  1988-09-25.txt       9   
3      George Bush  Vice President    REPUBLICAN  1988-09-25.txt       5   
4  Michael Dukakis        Governor      DEMOCRAT  1988-09-25.txt      22   

   Line_number  Sentiment  Verdict  
0           26   0.000000        0  
1           80  -0.740979        1  
2          129   

In [5]:
dataset.head()

,Text,labels
0,"You know, I saw a movie - ""Crocodile Dundee.""",0
1,We're consuming 50 percent of the world's coca...,1
2,That answer was about as clear as Boston harbor.,0
3,Let me help the governor.,0
4,We've run up more debt in the last eight years...,1


In [6]:
from datasets import Dataset, DatasetDict

# On transforme le DataFrame en Dataset Hugging Face
hg_dataset = Dataset.from_pandas(dataset[['Text', 'labels']])

# on divise notre dataset en train/test

split_dataset = hg_dataset.train_test_split(test_size=0.2)

# Partie Claim detecion

In [7]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import torch

model_checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize_function(examples):
    # L'option batched=True envoie une liste de textes ici
    return tokenizer(examples["Text"], padding="max_length", truncation=True)

tokenized_datasets = split_dataset.map(tokenize_function, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=2)

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    save_strategy="epoch",
    load_best_model_at_end=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
)

# Lancer l'entraînement
trainer.train()

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

KeyboardInterrupt: 

In [ ]:
trainer.save_model("./claim_detector_model")
tokenizer.save_pretrained("./claim_detector_model")

In [ ]:
from transformers import pipeline

claim_pipeline = pipeline("text-classification", model="./claim_detector_model", tokenizer="./claim_detector_model", device=0)


def detect_claim(text):
  results = claim_pipeline(text)

  is_claim = results[0]['label'] == 'LABEL_1'
  score = results[0]['score']

  return is_claim, score

In [ ]:
from evidence_retrieval import EvidenceRetriever

test_claim = {
    "The Eiffel Tower is in Paris": "en",
    "Le taux de chômage en France est de 7%": "fr",
    "La capital de España es Madrid": "es"
}

retriever = EvidenceRetriever(config_languages = {'en': 'en_core_web_sm', 'fr':'fr_core_news_sm', 'es':'es_core_news_sm'})

for claim, language in test_claim.items():
    evidence = retriever.get_evidence(claim, language)
    if evidence:
        print(f"✅ Trouvé ({language}) : {evidence['title']}")
        print(f"Extrait : {evidence['content'][:100]}...")
    else:
        print(f"❌ Non trouvé ({language}) : {claim}")

# Partie Claim Verification

In [ ]:
from claim_verification import ClaimVerifier

retriever = EvidenceRetriever(config_languages = {'en': 'en_core_web_sm', 'fr':'fr_core_news_sm', 'es':'es_core_news_sm'})
verifier = ClaimVerifier()

text_to_check = "The Eiffel Tower was built in 2012"

is_claim, detection_score = detect_claim(text_to_check)

if is_claim:
  print(f"🔍 Claim détecté ! (Confiance: {detection_score:.2f})")

  evidence = retriever.get_evidence(text_to_check, language='en')
  if evidence:
    verdict, confidence = verifier.verify(text_to_check, evidence['content'])
    print(f"\n--- RÉSULTAT FINAL ---")
    print(f"Phrase : {text_to_check}")
    print(f"Verdict : {verdict} (Score NLI: {confidence:.2f})")
    print(f"Source : {evidence['url']}")
  else:
        print("❌ Aucune preuve trouvée sur Wikipedia.")
else:
    print("ℹ️ Cette phrase n'est pas considérée comme une affirmation factuelle à vérifier.")